In [ ]:
text = "Hello how are you"

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2"
)


/Users/manojkrishnappa/github/FDE-AIML-COMPLETE-COURSE/.FDE-AIML-COMPLETE-COURSE/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 20:55:33.936000 88043 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18538.96it/s]


In [7]:
vector = embedding.embed_query(text)

In [8]:
len(vector)

384

In [9]:
vector[:10]

[-0.03483869880437851,
 0.06080599129199982,
 0.09206904470920563,
 0.04381448030471802,
 -0.04384098947048187,
 -0.08060234040021896,
 0.05162294954061508,
 0.0001953956380020827,
 -0.09116467833518982,
 0.0379410944879055]

In [10]:
vector = embedding.embed_query("Hello how are brother")

In [11]:
vector[:10]

[-0.035515788942575455,
 0.09207296371459961,
 0.03966245800256729,
 0.004840083885937929,
 -0.10010067373514175,
 -0.06820756196975708,
 0.051641013473272324,
 -0.03579706326127052,
 -0.060622431337833405,
 -0.03597932681441307]

In [12]:
vector = embedding.embed_query("ganesh is good boy")

In [13]:
vector[:10]

[-0.07142749428749084,
 0.040277354419231415,
 -0.00747288204729557,
 -0.05996067449450493,
 -0.07331134378910065,
 0.010595693252980709,
 0.04482664912939072,
 -0.04641513153910637,
 -0.009280853904783726,
 -0.039024803787469864]

In [1]:
# libraries

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# text loding lib
from langchain_community.document_loaders import TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# vector store
from langchain_community.vectorstores import Chroma

#prompt template and LCEL

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [2]:
# stage-1: Document loading stage

dir_loder = DirectoryLoader(
    "data",
    glob = "*.txt",
    loader_cls=TextLoader
)

documents = dir_loder.load()

# stage-2 Document splitting
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

In [5]:
chunks[0].metadata

{'source': 'data/machinelearning.txt'}

In [6]:
print(chunks[0].page_content)

Machine Learning (ML) is a core branch of Artificial Intelligence focused on building systems that learn from data.

Instead of relying on hardcoded instructions, ML models extract underlying patterns directly from observations.

The foundational goal of machine learning is to enable computers to make autonomous decisions without constant human intervention.

It combines principles from computer science, probability, linear algebra, and mathematical optimization.


In [8]:
#stage-3 embedings:
text = "Hello wwe are doiung embedings"

embedding = HuggingFaceEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2"
)


vector = embedding.embed_query(text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8190.14it/s]


In [9]:
vector[:10]

[-0.03927212953567505,
 -0.005548046436160803,
 -0.0022104443050920963,
 -0.027979731559753418,
 -0.007784606423228979,
 0.028514182195067406,
 -0.0345732718706131,
 0.025018634274601936,
 0.07547280937433243,
 -0.02120213583111763]

In [11]:
#stage-4 vectorstor

persistant_dir = "./chroma_db"

vectore_store = Chroma.from_documents(
    persist_directory = persistant_dir,
    documents = chunks,
    embedding=embedding,
    collection_name="FDE-quantamvectot"
)


In [13]:
query = "what is rag.?"

similar_docs = vectore_store.similarity_search(
    query,
    k=3
)
similar_docs

[Document(metadata={'source': 'data/rag.txt'}, page_content='Retrieval-Augmented Generation (RAG) is an artificial intelligence architecture \nthat optimizes large language model (LLM) responses by grounding them in external, \nauthoritative knowledge bases.   \nStandard LLMs rely solely on static training data, \nmaking them susceptible to factual errors and hallucinations. \nRAG addresses this through a three-step cycle:'),
 Document(metadata={'source': 'data/kubernetes.txt'}, page_content='applications run inside lightweight units called Pods.'),
 Document(metadata={'source': 'data/kubernetes.txt'}, page_content='load balancing, service discovery, and zero-downtime rolling updates. By abstracting the underlying physical or virtual cloud infrastructure, Kubernetes provides exceptional workload portability across public clouds, private data centers, and hybrid environments. Ultimately, it serves as the foundational operating system of the modern cloud-native ecosystem, enabling engine

In [15]:
query = "what is machine learning"

similar_docs = vectore_store.similarity_search_with_score(
    query,
    k=3
)
similar_docs

[(Document(metadata={'source': 'data/machinelearning.txt'}, page_content='Machine Learning (ML) is a core branch of Artificial Intelligence focused on building systems that learn from data.\n\nInstead of relying on hardcoded instructions, ML models extract underlying patterns directly from observations.\n\nThe foundational goal of machine learning is to enable computers to make autonomous decisions without constant human intervention.\n\nIt combines principles from computer science, probability, linear algebra, and mathematical optimization.'),
  0.35565871000289917),
 (Document(metadata={'source': 'data/machinelearning.txt'}, page_content='The concept originated in the mid-20th century, notably popularized by pioneer Arthur Samuel in 1959.\n\nToday, modern computing power and massive data availability have accelerated ML into everyday life.\n\nMachine learning is broadly divided into three primary categories: supervised, unsupervised, and reinforcement learning.\n\nSupervised learning

In [16]:
# stage-6 creating retriver

retriver = vectore_store.as_retriever(
    search_kwargs = {
        "k" :3 
    }
)

In [17]:
query = "what is machine learning"

retriver.invoke(query)

[Document(metadata={'source': 'data/machinelearning.txt'}, page_content='Machine Learning (ML) is a core branch of Artificial Intelligence focused on building systems that learn from data.\n\nInstead of relying on hardcoded instructions, ML models extract underlying patterns directly from observations.\n\nThe foundational goal of machine learning is to enable computers to make autonomous decisions without constant human intervention.\n\nIt combines principles from computer science, probability, linear algebra, and mathematical optimization.'),
 Document(metadata={'source': 'data/machinelearning.txt'}, page_content='The concept originated in the mid-20th century, notably popularized by pioneer Arthur Samuel in 1959.\n\nToday, modern computing power and massive data availability have accelerated ML into everyday life.\n\nMachine learning is broadly divided into three primary categories: supervised, unsupervised, and reinforcement learning.\n\nSupervised learning trains algorithms on labe

In [18]:
# stage-7 
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "qwen/qwen3.8-27b"
)

In [19]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_time': 0.019216759, 'completion_tokens_details': None, 'prompt_time': 0.000639749, 'prompt_tokens_details': None, 'queue_time': 0.05242217, 'total_time': 0.019856508}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1698f-4066-7f12-bc6e-517d6016cc64-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23})

In [20]:
cutom_prompt = ChatPromptTemplate.from_template(
    """ 
    you are helpfull assistant.

    answer the question using only provided context

    if the answer cannot be found in context or the retriver result then say
    "I dont know your question please ask only in the knowlede base" 

    Context:
    {context}

    Question:
    {question}

    Answer:

    """
)

In [24]:
# stage-9 building /chaining all the retriver prompt and also llm

ragchain = (
    {
        "context":retriver,
        "question":RunnablePassthrough()
    }
    | cutom_prompt
    | llm
    | StrOutputParser()

)

In [25]:
query = "what is machine learning"

In [27]:
answer = ragchain.invoke(query)
print(answer)

Machine Learning (ML) is a core branch of Artificial Intelligence focused on building systems that learn from data, enabling computers to make autonomous decisions without constant human intervention by extracting underlying patterns directly from observations.


In [28]:
query = " how was jailer 2 movie"
answer = ragchain.invoke(query)
print(answer)

I dont know your question please ask only in the knowlede base


In [ ]:
    print("Hello good Morning..")

Hello good Morning..
